In [1]:
%%capture

import altair as alt
import gcsfs
import pandas as pd

from IPython.display import HTML, Markdown, display
import prep_data_utils
from calitp_portfolio import magics

GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/ntd_explore/"

alt.data_transformers.enable("vegafusion")

# this one needs to be set somewhere, since the query will use it
min_year = 2018

In [2]:
# parameters cell for local
rtpa = "Metropolitan Transportation Commission"

In [3]:
%%capture_parameters
rtpa, min_year

{"rtpa": "Metropolitan Transportation Commission", "min_year": 2018}


In [4]:
# TODO: should the columns get subset?
# if publishing all columns, then we can leave it all here
not_published_cols = [
    "key", "legacy_ntd_id", "fta_region",
    "upt_prior_month", "upt_change_1mo", "upt_pct_change_1mo",
] 
df = pd.read_parquet(
    f"{GCS_FILE_PATH}monthly_with_crosswalk.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    filters = [[("rtpa", "==", rtpa)]]
).drop(columns = not_published_cols)

# no rows were found, but just in case
# does this need to affect the aggregations?
#df = df[(df.upt ==0) & (df.upt_change_1yr==0)].reset_index(drop=True)

# {rtpa}
## Monthly Ridership Trends

**Download data from our public [folder](https://console.cloud.google.com/storage/browser/calitp-publish-data-analysis)** by navigating to `ntd_monthly_ridership` and selecting a file.

Transit operators/agencies that are **Urban full reporters, that submit monthly ridership data to NTD from {min_year} to present**, are included in this report.

Operators/agencies that do not appear in the report may be due to:
- Were previously Urban full reporters, but are currently not 
- Non-monthly reporters (small system/rural/reduced reporters) 
- Has not reported data since 2018
- Has reported "0" data since 2018

Examples: 
- Reporter A is an urban full reporter from 2019-2022, then became a reduced reporter for 2023. Reporter A's ridership data will be displayed for 2019-2022 only.
- Reporter B is an urban full reporter from 2000-2017, then became a reduced reporter for 2018. Reporter B will not display ridership data.
- Reporter C was a reduced reporter form 2015-2020, then became an urban full reporter and began submitting monthly ridership data to NTD for 2021. Reporter C's ridership data will be displayed for 2021-present.

In [5]:
PUBLIC_FILENAME = df.month_first_day.max().strftime("%Y_%B")

URL = ("https://console.cloud.google.com/storage/"
       "browser/calitp-publish-data-analysis"
      )

display(
    HTML(
        f"""
        <a href={URL}>
        Download the latest month of data: {PUBLIC_FILENAME}</a>
        """
    )
)

In [6]:
# this is total upt since 2018, which is a parameter in the query
# might need to set this in update_vars, otherwise if it updates, 
# we don't know and caption is wrong
# agg by agency
agency_agg_yr = df.pipe(prep_data_utils.proportion_of_upt_by_agency)
total_upt = agency_agg_yr.total_upt.sum()
agency_count = agency_agg_yr.agency.nunique()

### Report Totals

In [7]:
Markdown(f"""
Within {rtpa}:
- Number of Reporters: <b>{agency_count}</b>.
- Total Unlinked Passenger Trips since {min_year}: <b>{total_upt:,}</b>.
- Individual Reporters ridership breakdown:
""")


Within Metropolitan Transportation Commission:
- Number of Reporters: <b>21</b>.
- Total Unlinked Passenger Trips since 2018: <b>2,742,428,023</b>.
- Individual Reporters ridership breakdown:


In [8]:
# new chart stuff - keep
# these chart sizes are different than annual
WIDTH = 325
HEIGHT = 150

color_scale = prep_data_utils.CALITP_CATEGORY_BRIGHT_COLORS + prep_data_utils.CALITP_CATEGORY_BOLD_COLORS


In [9]:
# Define all shared chart functions here
# annual has reporter_type, remove that for monthly
# tooltip switched for monthly

def title_by_group(group_col: str, y_col: str):
    """
    Set title here for consistency.
    """
    readable_group = group_col.replace("_", " ").replace("_full_name", "").title()

    if y_col=="upt":
        return f"Annual Unlinked Passenger Trips by {readable_group}"

    elif y_col =="upt_change_1yr":
        return  f"Yearly Change in Unlinked Passenger Trips by {readable_group}"

    
def tooltip_by_group(group_col: str): 
    """
    Consistent set of tooltip columns.
    """
    return ["month_first_day", "year", "month", "upt", "upt_change_1yr", group_col, "rtpa"]

In [10]:
def make_base_chart(
    df: pd.DataFrame,
    y_col: str,
    color_col: str,
) -> alt.Chart:
    """
    Use 1 base chart function. 
    year is always x-axis, make it ordinal for better display.
    tooltip is standardized with function to populate as much as we can.

    Everything else, such as title, even .mark_line(), .mark_bar() 
    can be layered on top of this function.
    """
    chart = (
        alt.Chart(df)
        .encode(
            x=alt.X("yearmonth(month_first_day):T", title="Date"),    
            y=alt.Y(
                y_col, title=y_col, 
                scale=alt.Scale(zero=False, clamp=True)
            ),
            color=alt.Color(
                color_col,
                scale=alt.Scale(range=color_scale),
                legend=None
            ),
            tooltip=tooltip_by_group(color_col),
        ).properties(width=WIDTH, height=HEIGHT)
        .interactive()
    )

    return chart

## Agency

In [11]:
agency_df = pd.read_parquet(
    f"{GCS_FILE_PATH}monthly/agency.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    filters = [[("rtpa", "==", rtpa)]]
)

In [12]:
make_base_chart(
    agency_df, 
    y_col = "upt", 
    color_col = "agency"
).mark_line().facet(
    "agency", columns = 2, title = ""
).properties(
    title=title_by_group("agency", "upt"),
).resolve_scale(x="independent", y="independent") 
# independent x-scale helps zooming for specific agencies, esp if we want to focus on a month
# annual report does x='shared'

alt.FacetChart(...)

In [13]:
make_base_chart(
    agency_df, 
    y_col="upt_change_1yr",
    color_col = "agency", 
).mark_bar().facet(
    "agency", columns = 2, title = ""
).properties(
    title={
        "text": title_by_group("agency", "upt_change_1yr"), 
        "subtitle": "Change in UPT from same month, prior year. (Jan 2026 compared to Jan 2025)"} ,
).resolve_scale(x="independent", y="independent")

alt.FacetChart(...)

### Transit Mode

In [14]:
mode_df = pd.read_parquet(
    f"{GCS_FILE_PATH}monthly/mode.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    filters = [[("rtpa", "==", rtpa)]]
)

In [15]:
make_base_chart(
    mode_df, 
    y_col = "upt", 
    color_col = "mode_full_name"
).mark_line().facet(
    "mode_full_name", columns = 2, title = ""
).properties(
    title=title_by_group("mode_full_name", "upt"),
).resolve_scale(x="independent", y="independent") 

alt.FacetChart(...)

In [16]:
make_base_chart(
    mode_df, 
    y_col="upt_change_1yr",
    color_col = "mode_full_name", 
).mark_bar().facet(
    "mode_full_name", columns = 2, title = ""
).properties(
    title={
        "text": title_by_group("mode_full_name", "upt_change_1yr"), 
        "subtitle": "Change in UPT from same month, prior year. (Jan 2026 compared to Jan 2025)"} ,
).resolve_scale(x="independent", y="independent")

alt.FacetChart(...)

### Type of Service

In [17]:
tos_df = pd.read_parquet(
    f"{GCS_FILE_PATH}monthly/type_of_service.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    filters = [[("rtpa", "==", rtpa)]]
)

In [18]:
make_base_chart(
    tos_df, 
    y_col = "upt", 
    color_col = "type_of_service_full_name"
).mark_line().facet(
    "type_of_service_full_name", columns = 2, title = ""
).properties(
    title=title_by_group("type_of_service_full_name", "upt"),
).resolve_scale(x="independent", y="independent") 

alt.FacetChart(...)

In [19]:
make_base_chart(
    tos_df, 
    y_col="upt_change_1yr",
    color_col = "type_of_service_full_name", 
).mark_bar().facet(
    "type_of_service_full_name", columns = 2, title = ""
).properties(
    title={
        "text": title_by_group("type_of_service_full_name", "upt_change_1yr"), 
        "subtitle": "Change in UPT from same month, prior year. (Jan 2026 compared to Jan 2025)"} ,
).resolve_scale(x="independent", y="independent")

alt.FacetChart(...)